# How AI Selects and Uses Tools Automatically

## Table of contents
- [What is tool selection?](#what-is-tool-selection)
- [Setup](#setup)
- [Step 1: Defining a tool (schema + handler)](#step-1-defining-a-tool-schema--handler)
- [Step 2: Tool choice — auto, any, specific](#step-2-tool-choice--auto-any-specific)
- [Step 3: The agentic loop (execution layer)](#step-3-the-agentic-loop-execution-layer)
- [Step 4: Binding schemas and handlers together](#step-4-binding-schemas-and-handlers-together)
- [Step 5: Full example with multiple tools](#step-5-full-example-with-multiple-tools)

## What is tool selection?

Claude is a **text-in / text-out** model. On its own it cannot fetch live data,
run calculations, or call external APIs. **Tools** bridge that gap.

When you give Claude a list of tools, it reads each tool's `name` and
`description`, then decides:

1. **Do I need a tool at all?** (Maybe I already know the answer.)
2. **Which tool fits this request?** (Based on the description.)
3. **What inputs does it need?** (Based on the `input_schema`.)

Claude then outputs a `tool_use` block — a structured JSON request — instead
of a plain text reply. Your code (the *execution layer*) receives that request,
calls the real function, and sends the result back. Claude then produces the
final answer.

```
User question
    ↓
Claude reads tool descriptions
    ↓ decides which tool to call
tool_use block { name, input }
    ↓
Your code runs the function
    ↓
tool_result sent back to Claude
    ↓
Claude writes final answer
```

> **Key insight**: Claude never executes code itself. It only outputs a
> structured request. *You* execute the tool and return the result.

## Setup

Ensure `@anthropic-ai/sdk` is installed and `ANTHROPIC_API_KEY` is in your
`.env` file.

In [ ]:
// npm install @anthropic-ai/sdk


In [ ]:
import Anthropic from 'npm:@anthropic-ai/sdk';
import { load } from 'https://deno.land/std@0.220.0/dotenv/mod.ts';

await load({ envPath: '../.env', export: true });

const MODEL_NAME = 'claude-sonnet-4-6';
const client = new Anthropic({ apiKey: Deno.env.get('ANTHROPIC_API_KEY') ?? '' });

console.log('Client ready. Model:', MODEL_NAME);


## Step 1: Defining a tool (schema + handler)

Every tool has two parts that you must keep together:

| Part | Purpose | Who uses it |
|---|---|---|
| **Schema** | Tells Claude what the tool is and what inputs it takes | Claude (decides whether/how to call it) |
| **Handler** | The actual TypeScript function that does the work | Your code (called after Claude requests it) |

### Writing a good schema

The `description` field is the most important part — it is Claude's only
signal for deciding whether to use this tool. Write it clearly:

- **Good**: `"Get the current temperature and weather conditions for a city."`
- **Bad**: `"Weather tool"`

Below we define a simple weather tool schema and its handler together.

In [ ]:
// --- SCHEMA: what Claude sees ---
const weatherSchema: Anthropic.Tool = {
  name: 'get_weather',
  description:
    'Get the current temperature and weather conditions for a city. ' +
    'Use this when the user asks about current weather.',
  input_schema: {
    type: 'object',
    properties: {
      city: { type: 'string', description: 'The city name, e.g. "Tokyo"' },
    },
    required: ['city'],
  },
};

// --- HANDLER: what your code runs ---
function getWeather(city: string): { city: string; temperature: number; condition: string } {
  // Mock data — replace with a real weather API call in production
  const data: Record<string, { temperature: number; condition: string }> = {
    Tokyo: { temperature: 28, condition: 'Partly cloudy' },
    London: { temperature: 15, condition: 'Rainy' },
    Sydney: { temperature: 22, condition: 'Sunny' },
  };
  return { city, ...(data[city] ?? { temperature: 20, condition: 'Unknown' }) };
}

console.log('Schema name:', weatherSchema.name);
console.log('Handler test:', getWeather('Tokyo'));


## Step 2: Tool choice — auto, any, specific

The `tool_choice` parameter controls *how* Claude is allowed to pick tools.

| Value | Behaviour |
|---|---|
| `{ type: 'auto' }` | Claude decides: use a tool **or** answer directly (default) |
| `{ type: 'any' }` | Claude **must** call one of the provided tools |
| `{ type: 'tool', name: '...' }` | Claude **must** call that exact tool |

### `auto` — Claude decides

With `auto`, Claude only calls a tool when it genuinely needs one.
Ask it something it knows → no tool. Ask about live data → tool.

In [ ]:
async function demoAutoChoice(question: string): Promise<void> {
  const response = await client.messages.create({
    model: MODEL_NAME,
    max_tokens: 512,
    tools: [weatherSchema],
    tool_choice: { type: 'auto' },
    messages: [{ role: 'user', content: question }],
  });

  const usedTool = response.stop_reason === 'tool_use';
  console.log(`Q: "${question}"`);
  console.log(`  stop_reason : ${response.stop_reason}`);
  console.log(`  used tool?  : ${usedTool}`);
  if (!usedTool) {
    const text = response.content.find(b => b.type === 'text');
    if (text?.type === 'text') console.log(`  answer      : ${text.text.slice(0, 80)}...`);
  }
  console.log();
}

// Claude knows this — should NOT call the tool
await demoAutoChoice('What is the capital of France?');

// Claude needs live data — SHOULD call the tool
await demoAutoChoice("What's the weather like in Tokyo right now?");


Notice how Claude skips the tool for general knowledge questions and calls it
only when the answer requires up-to-date external data.

### `any` — must pick one tool

`any` is useful when your application must always respond via a structured
tool result — for example, a chatbot that can only communicate by sending
a formatted message object.

In [ ]:
// Two tools: one for structured weather data, one for a plain text reply
const replySchema: Anthropic.Tool = {
  name: 'send_reply',
  description: 'Send a plain text reply to the user when no tool data is needed.',
  input_schema: {
    type: 'object',
    properties: { message: { type: 'string', description: 'The reply text' } },
    required: ['message'],
  },
};

const anyResponse = await client.messages.create({
  model: MODEL_NAME,
  max_tokens: 512,
  tools: [weatherSchema, replySchema],
  tool_choice: { type: 'any' },   // must call one of the two tools
  messages: [{ role: 'user', content: 'Hello! How are you?' }],
});

const chosenTool = anyResponse.content.find(b => b.type === 'tool_use');
if (chosenTool?.type === 'tool_use') {
  console.log('Tool chosen  :', chosenTool.name);
  console.log('Tool input   :', JSON.stringify(chosenTool.input, null, 2));
}


With `any`, Claude is forced to pick a tool. It chose `send_reply` because
the question did not require weather data — that is correct judgment.

### `tool` — force a specific tool

Use `{ type: 'tool', name: '...' }` when you always need structured output in
a fixed schema regardless of the question — for example, sentiment analysis.

In [ ]:
const sentimentSchema: Anthropic.Tool = {
  name: 'record_sentiment',
  description: 'Record the sentiment of a piece of text.',
  input_schema: {
    type: 'object',
    properties: {
      label: { type: 'string', enum: ['positive', 'neutral', 'negative'] },
      score: { type: 'number', description: 'Confidence 0.0–1.0' },
    },
    required: ['label', 'score'],
  },
};

const reviews = [
  'I absolutely love this product, it changed my life!',
  'Meh, it arrived on time I guess.',
  'Terrible quality, broke after one day. Awful.',
];

for (const review of reviews) {
  const res = await client.messages.create({
    model: MODEL_NAME,
    max_tokens: 256,
    tools: [sentimentSchema],
    tool_choice: { type: 'tool', name: 'record_sentiment' },  // always this tool
    messages: [{ role: 'user', content: `Analyze: "${review}"` }],
  });

  const toolBlock = res.content.find(b => b.type === 'tool_use');
  if (toolBlock?.type === 'tool_use') {
    const { label, score } = toolBlock.input as { label: string; score: number };
    console.log(`[${label.padEnd(8)} ${score.toFixed(2)}]  "${review.slice(0, 50)}"`);
  }
}


## Step 3: The agentic loop (execution layer)

A single `messages.create` call only returns a *request* from Claude — it
does **not** execute the tool. You need an **agentic loop**:

```
while (stop_reason === 'tool_use') {
  1. Find the tool_use block in response.content
  2. Call the matching handler function
  3. Append: assistant turn (tool_use block) + user turn (tool_result)
  4. Call messages.create again
}
// stop_reason === 'end_turn' → Claude has the answer
```

> **Why not let Claude call the function directly?**  
> Claude has no runtime. It cannot make HTTP requests, write files, or execute
> code. The loop above is the bridge between Claude's *intent* and your code's
> *action*.

Below is a minimal but complete loop:

In [ ]:
async function askWithTools(
  question: string,
  tools: Anthropic.Tool[],
  handlers: Map<string, (input: unknown) => unknown>,
): Promise<string> {
  const messages: Anthropic.MessageParam[] = [{ role: 'user', content: question }];

  while (true) {
    const response = await client.messages.create({
      model: MODEL_NAME,
      max_tokens: 1024,
      tools,
      messages,
    });

    if (response.stop_reason === 'end_turn') {
      // Claude is done — return the final text answer
      const text = response.content.find(b => b.type === 'text');
      return text?.type === 'text' ? text.text : '(no text response)';
    }

    if (response.stop_reason === 'tool_use') {
      // 1. Add assistant's response (contains the tool_use block)
      messages.push({ role: 'assistant', content: response.content });

      // 2. Execute every tool Claude requested this turn
      const toolResults: Anthropic.ToolResultBlockParam[] = [];
      for (const block of response.content) {
        if (block.type !== 'tool_use') continue;

        console.log(`  → Calling tool: ${block.name}(${JSON.stringify(block.input)})`);
        const handler = handlers.get(block.name);
        const result = handler ? handler(block.input) : { error: `Unknown tool: ${block.name}` };
        console.log(`  ← Result:`, result);

        toolResults.push({
          type: 'tool_result',
          tool_use_id: block.id,
          content: JSON.stringify(result),
        });
      }

      // 3. Send the results back so Claude can continue
      messages.push({ role: 'user', content: toolResults });
    }
  }
}

// Wire up schema → handler
const handlers = new Map<string, (input: unknown) => unknown>([
  ['get_weather', (input) => getWeather((input as { city: string }).city)],
]);

const answer = await askWithTools(
  "What's the weather like in London?",
  [weatherSchema],
  handlers,
);
console.log('\nFinal answer:', answer);


## Step 4: Binding schemas and handlers together

Passing schemas and handlers as separate structures works, but it's easy to
add a schema without a handler (or vice versa). A **`BoundTool`** type keeps
them together and makes the loop self-contained.

```
BoundTool = { name, description, input_schema, handler }
                ^--- everything Claude needs   ^--- everything your code needs
```

The `runWithTools` function below extracts schemas (for the API call) and
routes tool calls to handlers automatically — you never write the while-loop
again.

In [ ]:
// A tool definition that keeps schema and handler together
interface BoundTool {
  name: string;
  description: string;
  input_schema: Anthropic.Tool['input_schema'];
  handler: (input: unknown) => unknown;
}

// Runs the agentic loop; callers only deal with BoundTool[]
async function runWithTools(
  question: string,
  tools: BoundTool[],
  options?: { verbose?: boolean },
): Promise<string> {
  const schemas: Anthropic.Tool[] = tools.map(({ handler: _h, ...schema }) => schema);
  const handlerMap = new Map(tools.map(t => [t.name, t.handler]));
  const messages: Anthropic.MessageParam[] = [{ role: 'user', content: question }];

  while (true) {
    const response = await client.messages.create({
      model: MODEL_NAME,
      max_tokens: 1024,
      tools: schemas,
      messages,
    });

    if (response.stop_reason === 'end_turn') {
      const text = response.content.find(b => b.type === 'text');
      return text?.type === 'text' ? text.text : '(no text response)';
    }

    messages.push({ role: 'assistant', content: response.content });

    const toolResults: Anthropic.ToolResultBlockParam[] = [];
    for (const block of response.content) {
      if (block.type !== 'tool_use') continue;
      if (options?.verbose) console.log(`  [tool] ${block.name}(${JSON.stringify(block.input)})`);

      const handler = handlerMap.get(block.name);
      const result = handler ? handler(block.input) : { error: `Unknown tool: ${block.name}` };
      if (options?.verbose) console.log(`  [result]`, result);

      toolResults.push({
        type: 'tool_result',
        tool_use_id: block.id,
        content: JSON.stringify(result),
      });
    }
    messages.push({ role: 'user', content: toolResults });
  }
}

console.log('runWithTools helper ready.');


## Step 5: Full example with multiple tools

Now we put everything together. We define three tools — weather, calculator,
and a translation stub — all as `BoundTool` objects, then ask questions and
watch Claude pick the right tool automatically.

This is the core pattern you will use in any real application:

1. Define your tools as `BoundTool[]`
2. Pass the list to `runWithTools`
3. Let Claude decide which tool (or tools) to call

In [ ]:
// --- Define all tools as BoundTool objects ---

const weatherTool: BoundTool = {
  name: 'get_weather',
  description:
    'Get the current temperature and conditions for a city. ' +
    'Use this for questions about current weather.',
  input_schema: {
    type: 'object',
    properties: {
      city: { type: 'string', description: 'City name' },
    },
    required: ['city'],
  },
  handler: (input) => {
    const { city } = input as { city: string };
    return getWeather(city);
  },
};

const calculatorTool: BoundTool = {
  name: 'calculator',
  description:
    'Perform basic arithmetic: add, subtract, multiply, or divide two numbers. ' +
    'Use this whenever the user asks you to calculate or compute something.',
  input_schema: {
    type: 'object',
    properties: {
      a: { type: 'number', description: 'First number' },
      b: { type: 'number', description: 'Second number' },
      operation: {
        type: 'string',
        enum: ['add', 'subtract', 'multiply', 'divide'],
        description: 'Which arithmetic operation to apply',
      },
    },
    required: ['a', 'b', 'operation'],
  },
  handler: (input) => {
    const { a, b, operation } = input as { a: number; b: number; operation: string };
    if (operation === 'add') return { result: a + b };
    if (operation === 'subtract') return { result: a - b };
    if (operation === 'multiply') return { result: a * b };
    if (operation === 'divide') {
      if (b === 0) return { error: 'Division by zero' };
      return { result: a / b };
    }
    return { error: `Unknown operation: ${operation}` };
  },
};

const translateTool: BoundTool = {
  name: 'translate_text',
  description:
    'Translate a piece of text into a target language. ' +
    'Use this when the user asks you to translate something.',
  input_schema: {
    type: 'object',
    properties: {
      text: { type: 'string', description: 'The text to translate' },
      target_language: { type: 'string', description: 'Language to translate into, e.g. "Japanese"' },
    },
    required: ['text', 'target_language'],
  },
  handler: (input) => {
    // Mock translator — returns a labelled placeholder
    const { text, target_language } = input as { text: string; target_language: string };
    const translations: Record<string, Record<string, string>> = {
      Japanese: { 'Hello': 'こんにちは', 'Thank you': 'ありがとう' },
      French: { 'Hello': 'Bonjour', 'Thank you': 'Merci' },
    };
    const translated = translations[target_language]?.[text] ?? `[${target_language} translation of "${text}"]`;
    return { translated, target_language };
  },
};

const allTools: BoundTool[] = [weatherTool, calculatorTool, translateTool];

console.log('Tools registered:', allTools.map(t => t.name));


In [ ]:
// --- Ask questions and let Claude choose the right tool ---

const questions = [
  "What's the weather like in Sydney?",
  'What is 1234 multiplied by 56?',
  'Translate "Hello" into Japanese.',
  'What is the tallest mountain in the world?',  // Claude answers directly — no tool needed
];

for (const q of questions) {
  console.log(`\nQ: ${q}`);
  const answer = await runWithTools(q, allTools, { verbose: true });
  console.log(`A: ${answer}`);
  console.log('─'.repeat(60));
}


## Summary

Here is what you have learned:

| Concept | Key point |
|---|---|
| **Tool schema** | Defines what Claude *can* call; `description` drives selection |
| **Handler** | Your code that actually *runs* when Claude requests the tool |
| **`tool_choice: auto`** | Claude picks the right tool (or none) by reading descriptions |
| **`tool_choice: any`** | Forces Claude to always use a tool from the list |
| **`tool_choice: tool`** | Forces Claude to use one specific tool (great for structured output) |
| **Agentic loop** | `stop_reason === 'tool_use'` → execute → send `tool_result` → repeat |
| **BoundTool** | Keeps schema and handler together so they can't drift apart |

### Next steps

- **More tools**: Add real API calls (e.g. OpenWeatherMap) to replace the mock handlers
- **Tool search**: When you have dozens of tools, see
  `tool_search_alternate_approaches.ipynb` for how to let Claude *discover*
  tools on demand rather than loading them all upfront
- **Extended thinking + tools**: Combine tool use with Claude's extended
  reasoning — see `../thinking/extended_thinking_with_tool_use.ipynb`
- **Parallel tool calls**: Claude can call multiple tools in one turn; see
  the `parallel_tools.ipynb` cookbook in `claude-cookbooks`